# 002 SOT Modeling Credit Card Fraud

## Header

In [42]:
from notebook_config import setup

In [43]:
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config import settings

from src.ingestion.upload_dataset import BUCKET_KEY
from src.utils.s3_client import get_s3_client

In [44]:
import logging
logger = logging.getLogger(__name__)

## Dataset: Credit Card Fraud

In [45]:
bucket = settings.MINIO_BUCKET
key = BUCKET_KEY

s3 = get_s3_client()
obj = s3.get_object(Bucket=bucket, Key=BUCKET_KEY)
payload = obj['Body'].read()

df = pd.read_csv(io.BytesIO(payload))
df.head()

2026-05-01 17:31:02 | INFO | src.utils.s3_client | Creating S3 client with endpoint: http://localhost:9000
2026-05-01 17:31:02 | INFO | src.utils.s3_client | S3 client created successfully


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0000,-1.3598,-0.0728,2.5363,1.3782,-0.3383,0.4624,0.2396,0.0987,0.3638,0.0908,-0.5516,-0.6178,-0.9914,-0.3112,1.4682,-0.4704,0.2080,0.0258,0.4040,0.2514,-0.0183,0.2778,-0.1105,0.0669,0.1285,-0.1891,0.1336,-0.0211,149.6200,0
1,0.0000,1.1919,0.2662,0.1665,0.4482,0.0600,-0.0824,-0.0788,0.0851,-0.2554,-0.1670,1.6127,1.0652,0.4891,-0.1438,0.6356,0.4639,-0.1148,-0.1834,-0.1458,-0.0691,-0.2258,-0.6387,0.1013,-0.3398,0.1672,0.1259,-0.0090,0.0147,2.6900,0
2,1.0000,-1.3584,-1.3402,1.7732,0.3798,-0.5032,1.8005,0.7915,0.2477,-1.5147,0.2076,0.6245,0.0661,0.7173,-0.1659,2.3459,-2.8901,1.1100,-0.1214,-2.2619,0.5250,0.2480,0.7717,0.9094,-0.6893,-0.3276,-0.1391,-0.0554,-0.0598,378.6600,0
3,1.0000,-0.9663,-0.1852,1.7930,-0.8633,-0.0103,1.2472,0.2376,0.3774,-1.3870,-0.0550,-0.2265,0.1782,0.5078,-0.2879,-0.6314,-1.0596,-0.6841,1.9658,-1.2326,-0.2080,-0.1083,0.0053,-0.1903,-1.1756,0.6474,-0.2219,0.0627,0.0615,123.5000,0
4,2.0000,-1.1582,0.8777,1.5487,0.4030,-0.4072,0.0959,0.5929,-0.2705,0.8177,0.7531,-0.8228,0.5382,1.3459,-1.1197,0.1751,-0.4514,-0.2370,-0.0382,0.8035,0.4085,-0.0094,0.7983,-0.1375,0.1413,-0.2060,0.5023,0.2194,0.2152,69.9900,0


Definindo contrato de transformação SOR -> SOT.

In [46]:
df.columns

Index(['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10',
       'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20',
       'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount',
       'Class'],
      dtype='object')

In [53]:
EXPECTED_COLUMNS = ['Time'] + [f'V{i}' for i in range(1, 29)] + ['Amount', 'Class']
ALLOWED_CLASSES = {0, 1}

In [54]:
def validate_input(df):
    logger.info("Validando dados SOR")

    missing_cols = [c for c in EXPECTED_COLUMNS if c not in df.columns]
    extra_cols = [c for c in df.columns if c not in EXPECTED_COLUMNS]

    assert not missing_cols, f"Colunas faltando: {missing_cols}"
    assert not extra_cols, f"Colunas extras: {extra_cols}"
    logger.info("Colunas validadas")

    assert pd.api.types.is_numeric_dtype(df["Time"]), "Time deve ser numérico"
    assert pd.api.types.is_numeric_dtype(df["Amount"]), "Amount deve ser numérico"
    assert pd.api.types.is_numeric_dtype(df["Class"]), "Class deve ser numérico"
    logger.info("Tipos das Colunas validados")

    assert (df["Amount"] >= 0).all(), "Amount contém valores negativos"
    assert set(df["Class"].dropna().unique()).issubset(ALLOWED_CLASSES), "Class fora de {0,1}"
    logger.info("Valores das Colunas validados")

In [55]:
validate_input(df)

2026-05-01 17:33:47 | INFO | __main__ | Validando dados SOR
2026-05-01 17:33:47 | INFO | __main__ | Colunas validadas
2026-05-01 17:33:47 | INFO | __main__ | Tipos das Colunas validados
2026-05-01 17:33:47 | INFO | __main__ | Valores das Colunas validados


In [68]:
def build_sot(df) -> pd.DataFrame:

    sot_df: pd.DataFrame = df.copy()

    sot_df["transaction_id"] = pd.util.hash_pandas_object(
        sot_df[EXPECTED_COLUMNS], index=False
    ).astype("uint64").astype(str)
    logger.info("Coluna transaction_id criada")


    sot_df["amount_log"] = np.log1p(sot_df["Amount"]).astype("float64")
    logger.info("Coluna amount_log criada")

    sot_df["Time"] = sot_df["Time"].astype("int64")
    logger.info("Coluna Class tipada em int64")
    sot_df["Class"] = sot_df["Class"].astype("int8")
    logger.info("Coluna Class tipada em int8")
    sot_df["Amount"] = sot_df["Amount"].astype("float64")
    logger.info("Coluna Amount tipada em float64")
    sot_df["amount_log"] = sot_df["amount_log"].astype("float64")
    logger.info("Coluna amount_log tipada em float64")

    sot_df = sot_df.sort_values("Time").reset_index(drop=True)
    logger.info("DataFrame ordenado por Time")

    return sot_df

In [69]:
sot_df = build_sot(df)

2026-05-01 17:39:34 | INFO | __main__ | Coluna transaction_id criada
2026-05-01 17:39:34 | INFO | __main__ | Coluna amount_log criada
2026-05-01 17:39:34 | INFO | __main__ | Coluna Class tipada em int64
2026-05-01 17:39:34 | INFO | __main__ | Coluna Class tipada em int8
2026-05-01 17:39:34 | INFO | __main__ | Coluna Amount tipada em float64
2026-05-01 17:39:34 | INFO | __main__ | Coluna amount_log tipada em float64
2026-05-01 17:39:34 | INFO | __main__ | DataFrame ordenado por Time


In [70]:
sot_df

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class,transaction_id,amount_log
0,0,-1.3598,-0.0728,2.5363,1.3782,-0.3383,0.4624,0.2396,0.0987,0.3638,0.0908,-0.5516,-0.6178,-0.9914,-0.3112,1.4682,-0.4704,0.2080,0.0258,0.4040,0.2514,-0.0183,0.2778,-0.1105,0.0669,0.1285,-0.1891,0.1336,-0.0211,149.6200,0,11781865305817385170,5.0148
1,0,1.1919,0.2662,0.1665,0.4482,0.0600,-0.0824,-0.0788,0.0851,-0.2554,-0.1670,1.6127,1.0652,0.4891,-0.1438,0.6356,0.4639,-0.1148,-0.1834,-0.1458,-0.0691,-0.2258,-0.6387,0.1013,-0.3398,0.1672,0.1259,-0.0090,0.0147,2.6900,0,10742057486476263097,1.3056
2,1,-1.3584,-1.3402,1.7732,0.3798,-0.5032,1.8005,0.7915,0.2477,-1.5147,0.2076,0.6245,0.0661,0.7173,-0.1659,2.3459,-2.8901,1.1100,-0.1214,-2.2619,0.5250,0.2480,0.7717,0.9094,-0.6893,-0.3276,-0.1391,-0.0554,-0.0598,378.6600,0,8599885347999208214,5.9393
3,1,-0.9663,-0.1852,1.7930,-0.8633,-0.0103,1.2472,0.2376,0.3774,-1.3870,-0.0550,-0.2265,0.1782,0.5078,-0.2879,-0.6314,-1.0596,-0.6841,1.9658,-1.2326,-0.2080,-0.1083,0.0053,-0.1903,-1.1756,0.6474,-0.2219,0.0627,0.0615,123.5000,0,14235680070037929340,4.8243
4,2,-1.1582,0.8777,1.5487,0.4030,-0.4072,0.0959,0.5929,-0.2705,0.8177,0.7531,-0.8228,0.5382,1.3459,-1.1197,0.1751,-0.4514,-0.2370,-0.0382,0.8035,0.4085,-0.0094,0.7983,-0.1375,0.1413,-0.2060,0.5023,0.2194,0.2152,69.9900,0,10925877246138139066,4.2625
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
284802,172786,-11.8811,10.0718,-9.8348,-2.0667,-5.3645,-2.6068,-4.9182,7.3053,1.9144,4.3562,-1.5931,2.7119,-0.6893,4.6269,-0.9245,1.1076,1.9917,0.5106,-0.6829,1.4758,0.2135,0.1119,1.0145,-0.5093,1.4368,0.2500,0.9437,0.8237,0.7700,0,3437533384448522371,0.5710
284803,172787,-0.7328,-0.0551,2.0350,-0.7386,0.8682,1.0584,0.0243,0.2949,0.5848,-0.9759,-0.1502,0.9158,1.2148,-0.6751,1.1649,-0.7118,-0.0257,-1.2212,-1.5456,0.0596,0.2142,0.9244,0.0125,-1.0162,-0.6066,-0.3953,0.0685,-0.0535,24.7900,0,15153177278509151004,3.2500
284804,172788,1.9196,-0.3013,-3.2496,-0.5578,2.6305,3.0313,-0.2968,0.7084,0.4325,-0.4848,0.4116,0.0631,-0.1837,-0.5106,1.3293,0.1407,0.3135,0.3957,-0.5773,0.0014,0.2320,0.5782,-0.0375,0.6401,0.2657,-0.0874,0.0045,-0.0266,67.8800,0,11147728265506437243,4.2324
284805,172788,-0.2404,0.5305,0.7025,0.6898,-0.3780,0.6237,-0.6862,0.6791,0.3921,-0.3991,-1.9338,-0.9629,-1.0421,0.4496,1.9626,-0.6086,0.5099,1.1140,2.8978,0.1274,0.2652,0.8000,-0.1633,0.1232,-0.5692,0.5467,0.1088,0.1045,10.0000,0,8233480605528457250,2.3979
